In [31]:
import pandas as pd
import numpy as np
import os

In [2]:
df = pd.read_excel(r"..\Datos\Limpios\información_préstamos_limpio.xlsx")

df["Interes_Anual"] = df["Ratio_Interes"] / 100
df["i_mensual"] = df["Interes_Anual"] / 12
df["Ingresos_mensuales"] = df["Ingresos"] / 12

df.head()

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,...,Personas_Cargo,Proposito,Fiador,Impago,Prima,Duracion_anios,Ingresos_totales_prestamo,Interes_Anual,i_mensual,Ingresos_mensuales
0,ZLHEZX,25,30165,70653,664,40,2,10.33,48,0.55,...,0,Vivienda,0,0,699.49,4,120660,0.1033,0.008608,2513.750000
1,10SCEJ,19,20571,47545,847,8,3,21.79,36,0.55,...,1,Vivienda,1,1,626.04,3,61713,0.2179,0.018158,1714.250000
2,W9SHSZ,21,18766,40000,460,16,2,8.59,60,0.39,...,0,Vivienda,0,0,580.44,5,93830,0.0859,0.007158,1563.833333
3,5R0H8K,28,42900,43263,342,55,2,6.79,60,0.23,...,0,Vivienda,0,0,493.18,5,214500,0.0679,0.005658,3575.000000
4,8RP17L,26,42325,50756,358,20,1,10.74,48,0.26,...,0,Vivienda,0,0,537.82,4,169300,0.1074,0.008950,3527.083333


In [3]:
df.columns

Index(['ID', 'Edad', 'Ingresos', 'Monto_Inicial', 'Scoring_Crediticio',
       'Meses_Empleo', 'Num_Creditos', 'Ratio_Interes', 'Duracion',
       'Ratio_Deuda_Ingresos', 'Estudios', 'Tipo_Jornada_Laboral',
       'Estado_Civil', 'Posesion_Hipoteca', 'Personas_Cargo', 'Proposito',
       'Fiador', 'Impago', 'Prima', 'Duracion_anios',
       'Ingresos_totales_prestamo', 'Interes_Anual', 'i_mensual',
       'Ingresos_mensuales'],
      dtype='object')

In [4]:
def amortizacion_frances(capital, i, n):
    cuota = capital * (i * (1 + i)**n) / ((1 + i)**n - 1)
    saldo = capital
    intereses_totales = 0
    
    for _ in range(n):
        intereses = saldo * i
        amortizacion = cuota - intereses
        saldo -= amortizacion
        intereses_totales += intereses
        
    return cuota, intereses_totales

In [5]:
def amortizacion_aleman(capital, i, n):
    amortizacion_constante = capital / n
    saldo = capital
    intereses_totales = 0
    
    for _ in range(n):
        intereses = saldo * i
        saldo -= amortizacion_constante
        intereses_totales += intereses
        
    cuota_inicial = amortizacion_constante + capital * i
    
    return cuota_inicial, intereses_totales

In [6]:
cuota_frances = []
intereses_frances = []
cuota_aleman = []
intereses_aleman = []

for _, row in df.iterrows():
    
    C = row["Monto_Inicial"]
    i = row["i_mensual"]
    n = int(row["Duracion"])
    
    cf, int_f = amortizacion_frances(C, i, n)
    ca, int_a = amortizacion_aleman(C, i, n)
    
    cuota_frances.append(cf)
    intereses_frances.append(int_f)
    
    cuota_aleman.append(ca)
    intereses_aleman.append(int_a)

df["Cuota_Frances"] = cuota_frances
df["Intereses_Totales_Frances"] = intereses_frances
df["Cuota_Aleman_Inicial"] = cuota_aleman
df["Intereses_Totales_Aleman"] = intereses_aleman

df.head()

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,...,Prima,Duracion_anios,Ingresos_totales_prestamo,Interes_Anual,i_mensual,Ingresos_mensuales,Cuota_Frances,Intereses_Totales_Frances,Cuota_Aleman_Inicial,Intereses_Totales_Aleman
0,ZLHEZX,25,30165,70653,664,40,2,10.33,48,0.55,...,699.49,4,120660,0.1033,0.008608,2513.750000,1803.160490,15898.703542,2080.142075,14901.012088
1,10SCEJ,19,20571,47545,847,8,3,21.79,36,0.55,...,626.04,3,61713,0.2179,0.018158,1714.250000,1810.603909,17636.740720,2184.032403,15971.752229
2,W9SHSZ,21,18766,40000,460,16,2,8.59,60,0.39,...,580.44,5,93830,0.0859,0.007158,1563.833333,822.397411,9343.844652,953.000000,8733.166667
3,5R0H8K,28,42900,43263,342,55,2,6.79,60,0.23,...,493.18,5,214500,0.0679,0.005658,3575.000000,852.379313,7879.758806,965.846475,7466.292487
4,8RP17L,26,42325,50756,358,20,1,10.74,48,0.26,...,537.82,4,169300,0.1074,0.008950,3527.083333,1305.415920,11903.964172,1511.682867,11129.521900


In [7]:
#Metricas de riesgo
df["Ratio_Frances_Ingresos"] = df["Cuota_Frances"] / df["Ingresos_mensuales"]
df["Ratio_Aleman_Ingresos"] = df["Cuota_Aleman_Inicial"] / df["Ingresos_mensuales"]

df[["Ratio_Frances_Ingresos", "Ratio_Aleman_Ingresos"]].describe()

,Ratio_Frances_Ingresos,Ratio_Aleman_Ingresos
count,6639.000000,6639.000000
mean,0.648455,0.731916
std,0.228353,0.259580
min,0.174600,0.189589
25%,0.469502,0.529521
50%,0.622560,0.702992
75%,0.818804,0.917165
max,1.390491,1.695994


In [8]:
# CRITERIO DE ASIGNACIÓN

df["Diferencia_Intereses"] = (
    df["Intereses_Totales_Frances"] - df["Intereses_Totales_Aleman"]
)

df["Diferencia_Relativa"] = (
    df["Diferencia_Intereses"] / df["Monto_Inicial"]
)

scoring_umbral = df["Scoring_Crediticio"].quantile(0.70)

condicion_frances = (
    (df["Ratio_Frances_Ingresos"] <= 0.50) &         
    (df["Scoring_Crediticio"] >= scoring_umbral) &  
    ((df["Diferencia_Relativa"] <= 0.03))              
)

df["Sistema_Asignado"] = np.where(condicion_frances, "Frances", "Aleman")

df["Sistema_Asignado"].value_counts()

Sistema_Asignado
Aleman     6483
Frances     156
Name: count, dtype: int64

In [9]:
# Impacto real tras asignación

intereses_reales = np.where(
    df["Sistema_Asignado"] == "Frances",
    df["Intereses_Totales_Frances"],
    df["Intereses_Totales_Aleman"]
)

df["Intereses_Finales_Cartera"] = intereses_reales

print(f"Interés promedio real tras asignación: {df['Intereses_Finales_Cartera'].mean():.2f} €")

Interés promedio real tras asignación: 9190.71 €


In [10]:
ahorro_vs_frances = df["Intereses_Totales_Frances"].mean() - df["Intereses_Finales_Cartera"].mean()

print(f"Ahorro promedio frente a aplicar solo sistema francés: {ahorro_vs_frances:.2f} €")

Ahorro promedio frente a aplicar solo sistema francés: 784.50 €


In [11]:
tabla_comparacion = pd.DataFrame({
    "Sistema": ["Frances", "Aleman"],
    "Intereses_Promedio": [
        df["Intereses_Totales_Frances"].mean(),
        df["Intereses_Totales_Aleman"].mean()
    ],
    "Cuota_Promedio": [
        df["Cuota_Frances"].mean(),
        df["Cuota_Aleman_Inicial"].mean()
    ]
})

tabla_comparacion

,Sistema,Intereses_Promedio,Cuota_Promedio
0,Frances,9975.207728,1475.289269
1,Aleman,9178.921854,1657.793299


In [12]:
df[[
    "Monto_Inicial",
    "Duracion",
    "Cuota_Frances",
    "Cuota_Aleman_Inicial",
    "Intereses_Totales_Frances",
    "Intereses_Totales_Aleman",
    "Sistema_Asignado"
]].head()

,Monto_Inicial,Duracion,Cuota_Frances,Cuota_Aleman_Inicial,Intereses_Totales_Frances,Intereses_Totales_Aleman,Sistema_Asignado
0,70653,48,1803.160490,2080.142075,15898.703542,14901.012088,Aleman
1,47545,36,1810.603909,2184.032403,17636.740720,15971.752229,Aleman
2,40000,60,822.397411,953.000000,9343.844652,8733.166667,Aleman
3,43263,60,852.379313,965.846475,7879.758806,7466.292487,Aleman
4,50756,48,1305.415920,1511.682867,11903.964172,11129.521900,Aleman


In [13]:
df["Cuota_Frances"] = cuota_frances

In [14]:
df["Ratio_Frances_Ingresos"] = df["Cuota_Frances"] / df["Ingresos_mensuales"]

In [15]:
df["Sistema_Asignado"] = np.where(condicion_frances, "Frances", "Aleman")

In [16]:
df["Segmento_Riesgo"] = pd.cut(
    df["Ratio_Frances_Ingresos"],
    bins=[0,0.3,0.5,1,np.inf],
    labels=["Bajo","Medio","Alto","Muy alto"]
)

In [17]:
df.columns

Index(['ID', 'Edad', 'Ingresos', 'Monto_Inicial', 'Scoring_Crediticio',
       'Meses_Empleo', 'Num_Creditos', 'Ratio_Interes', 'Duracion',
       'Ratio_Deuda_Ingresos', 'Estudios', 'Tipo_Jornada_Laboral',
       'Estado_Civil', 'Posesion_Hipoteca', 'Personas_Cargo', 'Proposito',
       'Fiador', 'Impago', 'Prima', 'Duracion_anios',
       'Ingresos_totales_prestamo', 'Interes_Anual', 'i_mensual',
       'Ingresos_mensuales', 'Cuota_Frances', 'Intereses_Totales_Frances',
       'Cuota_Aleman_Inicial', 'Intereses_Totales_Aleman',
       'Ratio_Frances_Ingresos', 'Ratio_Aleman_Ingresos',
       'Diferencia_Intereses', 'Diferencia_Relativa', 'Sistema_Asignado',
       'Intereses_Finales_Cartera', 'Segmento_Riesgo'],
      dtype='object')

In [27]:
analisis_segmentos = df.groupby("Segmento_Riesgo").agg({
    "Monto_Inicial":"mean",
    "Cuota_Real":"mean",
    "VP_Cartera":"mean"
})
analisis_segmentos

C:\Users\eider\AppData\Local\Temp\ipykernel_38148\2707663638.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  analisis_segmentos = df.groupby("Segmento_Riesgo").agg({


,Monto_Inicial,Cuota_Real,VP_Cartera
Segmento_Riesgo,,,
Bajo,44456.129870,1035.827559,26972.720190
Medio,46732.718768,1266.402328,28604.440126
Alto,49879.039359,1744.458651,30690.749370
Muy alto,54027.593692,2496.624506,33387.804982


In [26]:
pd.crosstab(df["Segmento_Riesgo"], df["Sistema_Asignado"])

Sistema_Asignado,Aleman,Frances
Segmento_Riesgo,,
Bajo,229,2
Medio,1599,154
Alto,4116,0
Muy alto,539,0


In [20]:
prob_impago = df["Impago"].mean()

df["Riesgo_Frances"] = prob_impago * df["Monto_Inicial"]

In [21]:
df[["Monto_Inicial", "Riesgo_Frances"]].head()

,Monto_Inicial,Riesgo_Frances
0,70653,8056.080886
1,47545,5421.232866
2,40000,4560.927851
3,43263,4932.985540
4,50756,5787.361350


OBJETIVO 2

In [22]:
# VALOR PRESENTE DE LA CARTERA (objetivo 2)

df["k"] = (df["Duracion"] / 2).astype(int)
df["n_restante"] = df["Duracion"] - df["k"]

df["Cuota_Real"] = np.where(
    df["Sistema_Asignado"] == "Frances",
    df["Cuota_Frances"],
    df["Cuota_Aleman_Inicial"]
)

i = df["i_mensual"]

df["VP_Cartera"] = np.where(
    i == 0,
    df["Cuota_Real"] * df["n_restante"],
    df["Cuota_Real"] * (1 - (1+i)**(-df["n_restante"])) / i
)

print("Valor presente medio cartera:",
      round(df["VP_Cartera"].mean(),2),"€")

print("Valor presente total cartera:",
      round(df["VP_Cartera"].sum(),2),"€")

Valor presente medio cartera: 30229.47 €
Valor presente total cartera: 200693433.2 €


In [23]:
# SEGMENTACIÓN DE CLIENTES SEGÚN RATIO DE ENDEUDAMIENTO

df["Segmento_Riesgo"] = pd.cut(
    df["Ratio_Frances_Ingresos"],
    bins=[0,0.3,0.5,1,np.inf],
    labels=["Bajo","Medio","Alto","Muy alto"]
)

print("Distribución de clientes por riesgo:")
print(df["Segmento_Riesgo"].value_counts())

Distribución de clientes por riesgo:
Segmento_Riesgo
Alto        4116
Medio       1753
Muy alto     539
Bajo         231
Name: count, dtype: int64


In [24]:
analisis_segmentos = df.groupby("Segmento_Riesgo").agg({
    "Monto_Inicial":"mean",
    "Cuota_Real":"mean",
    "VP_Cartera":"mean"
})

analisis_segmentos

C:\Users\eider\AppData\Local\Temp\ipykernel_38148\1028058025.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  analisis_segmentos = df.groupby("Segmento_Riesgo").agg({


,Monto_Inicial,Cuota_Real,VP_Cartera
Segmento_Riesgo,,,
Bajo,44456.129870,1035.827559,26972.720190
Medio,46732.718768,1266.402328,28604.440126
Alto,49879.039359,1744.458651,30690.749370
Muy alto,54027.593692,2496.624506,33387.804982


In [26]:
pd.crosstab(df["Segmento_Riesgo"], df["Sistema_Asignado"])

Sistema_Asignado,Aleman,Frances
Segmento_Riesgo,,
Bajo,229,2
Medio,1599,154
Alto,4116,0
Muy alto,539,0


In [36]:
import plotly.express as px

fig1 = px.histogram(
    df,
    x="Monto_Inicial",
    nbins=30,
    title="Distribución del monto de los préstamos",
    labels={
        "Monto_Inicial": "Monto del préstamo",
        "count": "Frecuencia"
    },
    color_discrete_sequence=["#aa044c"]
)

fig1.show()

In [38]:
import plotly.graph_objects as go

intereses = [
    df["Intereses_Totales_Frances"].mean(),
    df["Intereses_Totales_Aleman"].mean()
]

fig = go.Figure(data=[
    go.Bar(
        x=["Sistema francés", "Sistema alemán"],
        y=intereses,
        marker_color="#aa044c"
    )
])

fig.update_layout(
    title="Comparación del interés total medio",
    xaxis_title="Sistema",
    yaxis_title="Intereses totales"
)

fig.show()

In [ ]:
import plotly.express as px

segmentos = df["Segmento_Riesgo"].value_counts().reset_index()
segmentos.columns = ["Segmento", "Cantidad"]

fig = px.bar(
    segmentos,
    x="Segmento",
    y="Cantidad",
    title="Distribución de clientes por segmento de riesgo",
    labels={
        "Segmento": "Segmento de riesgo",
        "Cantidad": "Número de clientes"
    },
    color_discrete_sequence=["#aa044c"]
)

fig.show()